In [ ]:
# ar/data-analysis/normal/06-missing-values
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## لماذا تهم القيم المفقودة

تحتوي كل مجموعة بيانات حقيقية تقريبًا على قيم مفقودة. إذا تجاهلتها، تعيد عمليات التجميع NaN، وتتعطل الرسوم البيانية، وتفشل نماذج التعلم الآلي. الخطوة الأولى في أي تحليل هي فهم ومعالجة البيانات المفقودة.


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")


## كشف القيم المفقودة

**فحص عمود واحد:**


In [ ]:
print(df["Age"].isna().sum())   # 177 missing Age values


**فحص جميع الأعمدة دفعة واحدة:**


In [ ]:
print(df.isna().sum())


المخرجات:


In [ ]:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


**رؤية النسبة المئوية المفقودة:**


In [ ]:
print((df.isna().sum() / len(df) * 100).round(1))


المخرجات:


In [ ]:
Cabin          77.1%
Age            19.9%
Embarked        0.2%
...


عمود الأماكن Cabin مفقود بنسبة 77% — أكثر من اللازم لملئه بشكل ذي معنى. العمر Age مفقود بنسبة 20% — يستحق محاولة ملئه. أما Embarked ففيه قيمةان مفقودتان فقط — سهل المعالجة.

## حذف القيم المفقودة

**حذف الصفوف التي تحتوي على أي قيم مفقودة:**


In [ ]:
df_clean = df.dropna()
print(df_clean.shape)   # (183, 12) — lost most rows


هذا عدواني جدًا لمعظم مجموعات البيانات. تفقد 708 من أصل 891 صفًا.

**حذف الصفوف التي تكون جميع قيمها مفقودة:**


In [ ]:
df_clean = df.dropna(how="all")


**حذف الصفوف ذات القيم المفقودة في أعمدة محددة:**


In [ ]:
df_clean = df.dropna(subset=["Age", "Embarked"])
print(df_clean.shape)   # (712, 12) — much better


**حذف الأعمدة التي تحتوي على عدد كبير جدًا من القيم المفقودة:**


In [ ]:
# Drop columns where more than 50% is missing
threshold = len(df) * 0.5
df_clean = df.dropna(thresh=threshold, axis=1)


## ملء القيم المفقودة

**الملء بثابت:**


In [ ]:
df["Embarked"] = df["Embarked"].fillna("S")   # most common port


**الملء بإحصائية:**


In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())


**الملء الأمامي أو الخلفي** — مفيد للسلاسل الزمنية:


In [ ]:
# Use the previous valid value to fill gaps
df["Price"] = df["Price"].ffill()

# Use the next valid value
df["Price"] = df["Price"].bfill()


**الملء بقيم مختلفة لكل عمود:**


In [ ]:
fill_values = {"Age": df["Age"].median(), "Embarked": "S", "Cabin": "Unknown"}
df = df.fillna(fill_values)


## اختيار استراتيجية

| السيناريو | الاستراتيجية |
|---|---|
| القيم المفقودة عشوائية وقليلة (< 5%) | الحذف باستخدام `dropna(subset=[...])` |
| قيم مفقودة في عمود رقمي | الملء بالوسيط (منيع ضد القيم الشاذة) |
| قيم مفقودة في عمود فئوي | الملء بالمنوال أو "قيمة غير معروفة" |
| العمود مفقود فيه أكثر من 50% | حذف العمود كاملًا |
| بيانات سلاسل زمنية | استخدام `ffill()` أو `bfill()` |

## مزالق شائعة

**الملء قبل تقسيم بيانات التدريب/الاختبار** — هذا يُسرّب معلومات. احسب قيم الملء على بيانات التدريب فقط، ثم طبقها على الاثنين.

**الحذف العدواني جدًا** — تحقق دائمًا من عدد الصفوف التي تخسرها. `dropna()` بدون وسائط يزيل غالبًا أكثر بكثير مما تتوقع.

**نسيان التحقق** — شغّل دائمًا `df.isna().sum()` بعد الملء للتأكد من عدم بقاء أي قيم NaN.

## جرّب بنفسك

من مجموعة بيانات تيتانيك:
1. احسب النسبة المئوية للقيم المفقودة لكل عمود
2. احذف عمود الـ Cabin (عدد كبير جدًا من القيم المفقودة)
3. املأ Age بمتوسط العمر
4. املأ Embarked بالقيمة الأكثر شيوعًا
5. تحقق من عدم بقاء قيم مفقودة


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

print((df.isna().sum() / len(df) * 100).round(1))

df = df.drop(columns=["Cabin"])
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isna().sum())


## خلاصات رئيسية

- افحص دائمًا القيم المفقودة أولًا باستخدام `isna().sum()` قبل اتخاذ قرار بشأن الاستراتيجية
- `dropna()` قوي لكنه غالبًا عدواني جدًا بدون `subset` أو `thresh`
- الملء بالوسيط أو المنوال عبر `fillna()` هو أكثر استراتيجيات الملء شيوعًا
- الأعمدة التي بها أكثر من 50% قيم مفقودة أفضل حالًا بحذفها من ملئها

## تحدي التطبيق

حمّل مجموعة بيانات تيتانيك وأنشئ نسخة نظيفة: احذف Cabin، واملأ Age بالوسيط، واملأ Embarked بالمنوال. ثم قارن معدل النجاة قبل التنظيف وبعده. هل غيّر التنظيف معدل النجاة الإجمالي؟ لماذا أو لماذا لا؟


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
